In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Phase 1

In [2]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

d = load_breast_cancer()
x = d.data
y = d.target

print("Baseline Model - Logistic Regression with StandardScaler")
print("=" * 50)

pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=10000, random_state=42))

sc = cross_val_score(pipe, x, y, cv=5, scoring='accuracy')

print(f"\nAccuracy per fold: {sc}")
print(f"Mean: {sc.mean():.4f}")
print(f"Std: {sc.std():.4f}")

Baseline Model - Logistic Regression with StandardScaler

Accuracy per fold: [0.98245614 0.98245614 0.97368421 0.97368421 0.99115044]
Mean: 0.9807
Std: 0.0065


Because we got really great outputs, I did not consider any preprocessing or other similar measures

# Phase 2 

In [3]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
import warnings
import time
warnings.filterwarnings('ignore')

data = load_breast_cancer()
X = data.data
y = data.target

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
metrics = ['accuracy', 'precision', 'recall']

print("Ensemble Models Comparison")
print("=" * 80)

dt = DecisionTreeClassifier(random_state=42)
bag = BaggingClassifier(estimator=dt, n_estimators=50, random_state=42)

start = time.time()
bag_results = cross_validate(bag, X, y, cv=skf, scoring=metrics)
bag_time = time.time() - start

print("\n1. BaggingClassifier (base: DecisionTreeClassifier)")
print(f"   Accuracy:  {bag_results['test_accuracy'].mean():.4f} ± {bag_results['test_accuracy'].std():.4f}")
print(f"   Precision: {bag_results['test_precision'].mean():.4f}")
print(f"   Recall:    {bag_results['test_recall'].mean():.4f}")
print(f"   Time:      {bag_time:.4f}s")

stump = DecisionTreeClassifier(max_depth=1, random_state=42)
ada = AdaBoostClassifier(estimator=stump, n_estimators=50, random_state=42, algorithm='SAMME')

start = time.time()
ada_results = cross_validate(ada, X, y, cv=skf, scoring=metrics)
ada_time = time.time() - start

print("\n2. AdaBoostClassifier (base: Decision Stump)")
print(f"   Accuracy:  {ada_results['test_accuracy'].mean():.4f} ± {ada_results['test_accuracy'].std():.4f}")
print(f"   Precision: {ada_results['test_precision'].mean():.4f}")
print(f"   Recall:    {ada_results['test_recall'].mean():.4f}")
print(f"   Time:      {ada_time:.4f}s")

rf = RandomForestClassifier(n_estimators=100, random_state=42)

start = time.time()
rf_results = cross_validate(rf, X, y, cv=skf, scoring=metrics)
rf_time = time.time() - start

print("\n3. RandomForestClassifier")
print(f"   Accuracy:  {rf_results['test_accuracy'].mean():.4f} ± {rf_results['test_accuracy'].std():.4f}")
print(f"   Precision: {rf_results['test_precision'].mean():.4f}")
print(f"   Recall:    {rf_results['test_recall'].mean():.4f}")
print(f"   Time:      {rf_time:.4f}s")

print("\n" + "=" * 80)
print("\nComparison Table:")
print(f"{'Model':<30} {'Accuracy':<20} {'Precision':<12} {'Recall':<12} {'Time':<10}")
print("-" * 80)
print(f"{'BaggingClassifier':<30} {bag_results['test_accuracy'].mean():.4f} ± {bag_results['test_accuracy'].std():.4f}    {bag_results['test_precision'].mean():.4f}      {bag_results['test_recall'].mean():.4f}      {bag_time:.4f}s")
print(f"{'AdaBoostClassifier':<30} {ada_results['test_accuracy'].mean():.4f} ± {ada_results['test_accuracy'].std():.4f}    {ada_results['test_precision'].mean():.4f}      {ada_results['test_recall'].mean():.4f}      {ada_time:.4f}s")
print(f"{'RandomForestClassifier':<30} {rf_results['test_accuracy'].mean():.4f} ± {rf_results['test_accuracy'].std():.4f}    {rf_results['test_precision'].mean():.4f}      {rf_results['test_recall'].mean():.4f}      {rf_time:.4f}s")


Ensemble Models Comparison

1. BaggingClassifier (base: DecisionTreeClassifier)
   Accuracy:  0.9578 ± 0.0140
   Precision: 0.9619
   Recall:    0.9720
   Time:      2.1375s

2. AdaBoostClassifier (base: Decision Stump)
   Accuracy:  0.9525 ± 0.0219
   Precision: 0.9570
   Recall:    0.9692
   Time:      1.2165s

3. RandomForestClassifier
   Accuracy:  0.9561 ± 0.0123
   Precision: 0.9651
   Recall:    0.9665
   Time:      1.3626s


Comparison Table:
Model                          Accuracy             Precision    Recall       Time      
--------------------------------------------------------------------------------
BaggingClassifier              0.9578 ± 0.0140    0.9619      0.9720      2.1375s
AdaBoostClassifier             0.9525 ± 0.0219    0.9570      0.9692      1.2165s
RandomForestClassifier         0.9561 ± 0.0123    0.9651      0.9665      1.3626s


# Phase 3

In [4]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

data = load_breast_cancer()
X = data.data
y = data.target

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr = LogisticRegression(max_iter=10000, random_state=42)
lr_scores = cross_val_score(lr, X, y, cv=skf, scoring='accuracy')

dt = DecisionTreeClassifier(random_state=42)
bag = BaggingClassifier(estimator=dt, n_estimators=50, random_state=42)
bag_scores = cross_val_score(bag, X, y, cv=skf, scoring='accuracy')

stump = DecisionTreeClassifier(max_depth=1, random_state=42)
ada = AdaBoostClassifier(estimator=stump, n_estimators=50, random_state=42, algorithm='SAMME')
ada_scores = cross_val_score(ada, X, y, cv=skf, scoring='accuracy')

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_scores = cross_val_score(rf, X, y, cv=skf, scoring='accuracy')

alpha = 0.05

print("Paired t-test Results (Baseline vs Ensemble Methods)")
print("=" * 70)
print(f"Baseline (LogisticRegression) Accuracy: {lr_scores.mean():.4f}")
print("\n")

t_stat_bag, p_val_bag = stats.ttest_rel(bag_scores, lr_scores)
print(f"1. BaggingClassifier vs Baseline")
print(f"   BaggingClassifier Accuracy: {bag_scores.mean():.4f}")
print(f"   t-statistic: {t_stat_bag:.4f}")
print(f"   p-value: {p_val_bag:.4f}")
if p_val_bag < alpha:
    if bag_scores.mean() > lr_scores.mean():
        print(f"   Result: BaggingClassifier is significantly BETTER (p < {alpha})")
    else:
        print(f"   Result: BaggingClassifier is significantly WORSE (p < {alpha})")
else:
    print(f"   Result: No significant difference (p >= {alpha})")

print("\n")

t_stat_ada, p_val_ada = stats.ttest_rel(ada_scores, lr_scores)
print(f"2. AdaBoostClassifier vs Baseline")
print(f"   AdaBoostClassifier Accuracy: {ada_scores.mean():.4f}")
print(f"   t-statistic: {t_stat_ada:.4f}")
print(f"   p-value: {p_val_ada:.4f}")
if p_val_ada < alpha:
    if ada_scores.mean() > lr_scores.mean():
        print(f"   Result: AdaBoostClassifier is significantly BETTER (p < {alpha})")
    else:
        print(f"   Result: AdaBoostClassifier is significantly WORSE (p < {alpha})")
else:
    print(f"   Result: No significant difference (p >= {alpha})")

print("\n")

t_stat_rf, p_val_rf = stats.ttest_rel(rf_scores, lr_scores)
print(f"3. RandomForestClassifier vs Baseline")
print(f"   RandomForestClassifier Accuracy: {rf_scores.mean():.4f}")
print(f"   t-statistic: {t_stat_rf:.4f}")
print(f"   p-value: {p_val_rf:.4f}")
if p_val_rf < alpha:
    if rf_scores.mean() > lr_scores.mean():
        print(f"   Result: RandomForestClassifier is significantly BETTER (p < {alpha})")
    else:
        print(f"   Result: RandomForestClassifier is significantly WORSE (p < {alpha})")
else:
    print(f"   Result: No significant difference (p >= {alpha})")

print("\n" + "=" * 70)
print("\nSummary Table:")
print(f"{'Model':<25} {'Accuracy':<12} {'t-stat':<10} {'p-value':<10} {'Result'}")
print("-" * 70)
print(f"{'Baseline (LR)':<25} {lr_scores.mean():.4f}       -          -          -")
print(f"{'BaggingClassifier':<25} {bag_scores.mean():.4f}       {t_stat_bag:.4f}     {p_val_bag:.4f}     {'Sig.' if p_val_bag < alpha else 'Not Sig.'}")
print(f"{'AdaBoostClassifier':<25} {ada_scores.mean():.4f}       {t_stat_ada:.4f}     {p_val_ada:.4f}     {'Sig.' if p_val_ada < alpha else 'Not Sig.'}")
print(f"{'RandomForestClassifier':<25} {rf_scores.mean():.4f}       {t_stat_rf:.4f}     {p_val_rf:.4f}     {'Sig.' if p_val_rf < alpha else 'Not Sig.'}")


Paired t-test Results (Baseline vs Ensemble Methods)
Baseline (LogisticRegression) Accuracy: 0.9543


1. BaggingClassifier vs Baseline
   BaggingClassifier Accuracy: 0.9578
   t-statistic: 0.7791
   p-value: 0.4794
   Result: No significant difference (p >= 0.05)


2. AdaBoostClassifier vs Baseline
   AdaBoostClassifier Accuracy: 0.9525
   t-statistic: -0.2109
   p-value: 0.8433
   Result: No significant difference (p >= 0.05)


3. RandomForestClassifier vs Baseline
   RandomForestClassifier Accuracy: 0.9561
   t-statistic: 0.4082
   p-value: 0.7040
   Result: No significant difference (p >= 0.05)


Summary Table:
Model                     Accuracy     t-stat     p-value    Result
----------------------------------------------------------------------
Baseline (LR)             0.9543       -          -          -
BaggingClassifier         0.9578       0.7791     0.4794     Not Sig.
AdaBoostClassifier        0.9525       -0.2109     0.8433     Not Sig.
RandomForestClassifier    0.9561   